In [1]:
import pandas as pd

In [49]:
data = pd.read_csv('data/DrugAge/drugage.csv')
data['dosage'] = data['dosage'].astype(str)

In [50]:
data.nunique().to_dict()

{'compound_name': 1046,
 'species': 38,
 'strain': 158,
 'dosage': 720,
 'age_at_initiation': 48,
 'treatment_duration': 14,
 'avg_lifespan_change_percent': 1493,
 'avg_lifespan_significance': 2,
 'max_lifespan_change_percent': 395,
 'max_lifespan_significance': 2,
 'gender': 5,
 'weight_change_percent': 47,
 'weight_change_significance': 2,
 'ITP': 2,
 'pubmed_id': 664}

In [51]:
dosage_list = data.dosage.unique().tolist()

In [52]:
[x for x in dosage_list if 'mM' in x]

['0.0001 mM',
 '0.001 mM',
 '0.01 mM',
 '0.1 mM',
 '1 mM',
 '5 mM',
 '0.5 mM',
 '2 mM water',
 '0.4 mM water',
 '0.08 mM water',
 '0.45 mM',
 '3 mM',
 '60 mM',
 '100 mM',
 '10 mM',
 '500 mM',
 '2 mM',
 '17 mM',
 '20 mM',
 '0.05 mM',
 '2.4 mM',
 '1.2 mM',
 '250 mM',
 '0.25 mM',
 '25 mM',
 '50 mM',
 '0.016 mM',
 '0.0016 mM',
 '6 mM',
 '0.2-0.4 mM',
 '8 mM',
 '0.2 mM',
 '4 mM',
 '2.5 mM',
 '0.1 mM/l',
 '9 mM',
 '0.3 mM',
 '200 mM',
 '0.87 mM',
 '4.5 mM',
 '7.5 mM',
 '30 mM',
 '75 mM',
 '274 mM',
 '6.25 mM',
 '12.5 mM',
 '0.4 mM',
 '0.025 mM',
 '900 mM',
 '15 mM',
 '45 mM',
 '350 mM',
 '150 mM',
 '3.18 mM',
 '2.54 mM',
 '111 mM',
 '555 mM',
 '55 mM',
 '400 mM',
 '3 mM water',
 '0.3mM water',
 '0.36mM',
 '0.05mM',
 '1.25 mM',
 '0.5mM',
 '300 mM',
 '100mM',
 '2mM',
 '600 mM',
 '0.0083 mM']

In [53]:
from collections import Counter

In [54]:
simple_entries = list(filter(lambda x: len(str(x).split())==2, dosage_list))

In [55]:
Counter([str(x).split()[-1] for x in simple_entries])

Counter({'µM': 77,
         'mM': 59,
         'µg/ml': 33,
         'μM': 32,
         'mg/mL': 26,
         'mg/ml': 23,
         'M': 17,
         'µg/mL': 17,
         'nM': 15,
         'ppm': 11,
         'μg/mL': 11,
         'µg': 11,
         'mg/L': 9,
         'mg/kg': 8,
         'g/L': 8,
         'mg/kg/week': 6,
         'IU': 6,
         '%': 5,
         'mg/l': 5,
         'μg/ml': 5,
         'µ/mL': 5,
         'mg/kg/day': 4,
         'nmol': 4,
         'ppb': 4,
         'ng/ml': 4,
         '2x/week': 3,
         '(w/v)': 3,
         'µm': 3,
         'µmol/L': 3,
         'g/kg': 3,
         'mg/g': 3,
         'mg/plate': 3,
         'wt/vol': 2,
         'ug/ml': 2,
         'μm': 2,
         'w/v': 2,
         'ml': 2,
         'pM': 2,
         'ppm/day': 1,
         'mg': 1,
         'g': 1,
         'dilution': 1,
         'mg/day': 1,
         'mM/l': 1,
         'drop': 1,
         'uM': 1,
         'µL': 1,
         'gr': 1,
         'µmol/l': 1,
      

In [84]:
import re
from dataclasses import dataclass
from typing import Optional


@dataclass
class ParsedDose:
    raw: str
    kept: bool
    keep_reason: str                # simple | medium | frequency | reject_*
    value: Optional[float] = None
    unit: Optional[str] = None
    medium: Optional[str] = None    # food | water | diet | medium
    frequency: Optional[str] = None # e.g. 2_per_week, every_5_days


UNITS = [
    "mg/kg/week",
    "mg/kg/day",
    "mg/kg",
    "ug/mL",
    "ug/L",
    "mg/mL",
    "mg/L",
    "ug/g",
    "g/kg",
    "g/L",
    "ppm/day",
    "ppm",
    "mM",
    "uM",
    "nM",
    "pM",
    "mg",
    "ug",
    "IU",
    "M",
    "%",
]
UNIT_PATTERN = "|".join(re.escape(u) for u in sorted(UNITS, key=len, reverse=True))

STOPWORDS = {"on", "in", "at", "the", "of", "and", "with", "for", "to"}


def normalize_dose_text(x) -> str:
    if x is None:
        return ""
    x = str(x).strip()
    x = re.sub(r"\([^)]*\)", "", x).strip()
    x = x.replace("μ", "u").replace("µ", "u")
    x = x.replace("\u2009", " ").replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    x = x.replace("mg/l", "mg/L").replace("ug/ml", "ug/mL").replace("mg/ml", "mg/mL")
    x = x.replace("µg/ml", "ug/mL").replace("μg/ml", "ug/mL")
    x = x.replace("times per week", "times a week")
    return x


def parse_frequency(text: str) -> Optional[str]:
    t = text.lower()

    m = re.search(r"(\d+(?:\.\d+)?)\s*x\s*/\s*(day|week|month)\b", t)
    if m:
        return f"{m.group(1)}_per_{m.group(2)}"

    m = re.search(r"(\d+(?:\.\d+)?)\s*times a\s*(day|week|month)\b", t)
    if m:
        return f"{m.group(1)}_per_{m.group(2)}"

    m = re.search(r"every\s+(\d+(?:\.\d+)?)\s+(day|days|week|weeks|month|months)\b", t)
    if m:
        unit = m.group(2).rstrip("s")
        return f"every_{m.group(1)}_{unit}s"

    if "alternate days" in t or "every other day" in t:
        return "every_2_days"

    return None


def parse_medium(text: str) -> Optional[str]:
    t = text.lower()
    for medium in ["food", "water", "diet", "medium"]:
        if re.search(rf"\b{medium}\b", t):
            return medium
    return None


def has_reject_pattern(text: str) -> Optional[str]:
    t = text.lower()

    if " plus " in t:
        return "reject_combination"
    if "," in t and re.search(r"\d", t) and re.search(r"\b(month|months|day|days|week|weeks|death)\b", t):
        return "reject_multiphase"
    if re.search(r"\b\d+(?:\.\d+)?x10\^-?\d+\b", t):
        return "reject_ratio_like"
    if re.search(r"/\s*(food|diet|water|medium)\b", t):
        return "reject_ratio_like"
    if "bodyweight" in t:
        return "reject_bodyweight"
    if any(route in t for route in [
        "intraperitoneal",
        "oral gavage",
        "subcutaneous injection",
        "injection",
        "orally",
        " oral ",
    ]):
        return "reject_route"
    return None


def cleanup_remainder(text: str, value: float, unit: str, medium: Optional[str], frequency: Optional[str]) -> str:
    rem = text

    rem = re.sub(rf"\b{re.escape(str(value))}\s*{re.escape(unit)}\b", " ", rem, flags=re.IGNORECASE)

    if medium:
        rem = re.sub(rf"\b{medium}\b", " ", rem, flags=re.IGNORECASE)

    rem = re.sub(r"(\d+(?:\.\d+)?)\s*x\s*/\s*(day|week|month)\b", " ", rem, flags=re.IGNORECASE)
    rem = re.sub(r"(\d+(?:\.\d+)?)\s*times a\s*(day|week|month)\b", " ", rem, flags=re.IGNORECASE)
    rem = re.sub(r"every\s+(\d+(?:\.\d+)?)\s+(day|days|week|weeks|month|months)\b", " ", rem, flags=re.IGNORECASE)
    rem = re.sub(r"\b(alternate days|every other day)\b", " ", rem, flags=re.IGNORECASE)

    rem = re.sub(r"[\s,;()]+", " ", rem).strip().lower()

    toks = [tok for tok in rem.split() if tok not in STOPWORDS]
    return " ".join(toks)


def parse_dose(raw_value) -> ParsedDose:
    raw = normalize_dose_text(raw_value)
    if not raw:
        return ParsedDose(raw=raw, kept=False, keep_reason="reject_empty")

    reject_reason = has_reject_pattern(raw)
    if reject_reason:
        return ParsedDose(raw=raw, kept=False, keep_reason=reject_reason)

    m = re.fullmatch(
        rf"\s*(\d+(?:\.\d+)?)\s*({UNIT_PATTERN})(?:\s+(food|water|diet|medium))?\s*",
        raw,
        flags=re.IGNORECASE,
    )
    if m:
        value = float(m.group(1))
        unit = m.group(2)
        medium = m.group(3)
        return ParsedDose(
            raw=raw,
            kept=True,
            keep_reason="medium" if medium else "simple",
            value=value,
            unit=unit,
            medium=medium,
        )

    m = re.search(rf"\b(\d+(?:\.\d+)?)\s*({UNIT_PATTERN})(?![A-Za-z/])", raw, flags=re.IGNORECASE)
    if not m:
        return ParsedDose(raw=raw, kept=False, keep_reason="reject_no_amount_unit")

    value = float(m.group(1))
    unit = m.group(2)
    medium = parse_medium(raw)
    frequency = parse_frequency(raw)

    remainder = cleanup_remainder(raw, value, unit, medium, frequency)
    if remainder:
        return ParsedDose(
            raw=raw,
            kept=False,
            keep_reason="reject_partial",
            value=value,
            unit=unit,
            medium=medium,
            frequency=frequency,
        )

    if frequency:
        return ParsedDose(
            raw=raw,
            kept=True,
            keep_reason="frequency" if not medium else "medium_frequency",
            value=value,
            unit=unit,
            medium=medium,
            frequency=frequency,
        )

    if medium:
        return ParsedDose(
            raw=raw,
            kept=True,
            keep_reason="medium",
            value=value,
            unit=unit,
            medium=medium,
        )

    return ParsedDose(
        raw=raw,
        kept=True,
        keep_reason="simple",
        value=value,
        unit=unit,
    )

In [85]:
dose_objects = list(map(parse_dose, dosage_list))

In [86]:
dose_objects

[ParsedDose(raw='4%', kept=True, keep_reason='simple', value=4.0, unit='%', medium=None, frequency=None),
 ParsedDose(raw='2%', kept=True, keep_reason='simple', value=2.0, unit='%', medium=None, frequency=None),
 ParsedDose(raw='0.0001 mM', kept=True, keep_reason='simple', value=0.0001, unit='mM', medium=None, frequency=None),
 ParsedDose(raw='0.001 mM', kept=True, keep_reason='simple', value=0.001, unit='mM', medium=None, frequency=None),
 ParsedDose(raw='0.01 mM', kept=True, keep_reason='simple', value=0.01, unit='mM', medium=None, frequency=None),
 ParsedDose(raw='0.1 mM', kept=True, keep_reason='simple', value=0.1, unit='mM', medium=None, frequency=None),
 ParsedDose(raw='1 mM', kept=True, keep_reason='simple', value=1.0, unit='mM', medium=None, frequency=None),
 ParsedDose(raw='5 mM', kept=True, keep_reason='simple', value=5.0, unit='mM', medium=None, frequency=None),
 ParsedDose(raw='0.5 mM', kept=True, keep_reason='simple', value=0.5, unit='mM', medium=None, frequency=None),
 Pa

In [87]:
Counter([x.kept for x in dose_objects])

Counter({True: 538, False: 182})

In [89]:
[x for x in dose_objects if not x.kept]

[ParsedDose(raw='0.5 mg, 5 times a week', kept=False, keep_reason='reject_multiphase', value=None, unit=None, medium=None, frequency=None),
 ParsedDose(raw='0.25% on alternate days', kept=False, keep_reason='reject_partial', value=0.25, unit='%', medium=None, frequency='every_2_days'),
 ParsedDose(raw='3-3.2 ppm water', kept=False, keep_reason='reject_partial', value=3.2, unit='ppm', medium='water', frequency=None),
 ParsedDose(raw='0.0025', kept=False, keep_reason='reject_no_amount_unit', value=None, unit=None, medium=None, frequency=None),
 ParsedDose(raw='0.000001', kept=False, keep_reason='reject_no_amount_unit', value=None, unit=None, medium=None, frequency=None),
 ParsedDose(raw='0.1 mg subcutaneous injection 5 consecutive days monthly', kept=False, keep_reason='reject_route', value=None, unit=None, medium=None, frequency=None),
 ParsedDose(raw='270 ppm water plus 289 ppm food', kept=False, keep_reason='reject_combination', value=None, unit=None, medium=None, frequency=None),
 Pa

In [90]:
print(data.sample(10))

                   compound_name                  species       strain  \
3174       Propantheline Bromide   Caenorhabditis elegans           N2   
0                        Ethanol    Drosophila mojavensis         A300   
2613                  Metamizole   Caenorhabditis elegans           N2   
2829  Epigallocatechin-3-gallate   Caenorhabditis elegans           N2   
827                  Resveratrol  Drosophila melanogaster      Dahomey   
2933            Choline Chloride  Drosophila melanogaster     Canton-S   
1315                     Tyrosol   Caenorhabditis elegans  fer-15(b26)   
3308        7,8-Dihydroxyflavone   Caenorhabditis elegans           N2   
1636            Turmeric extract  Drosophila melanogaster          NaN   
15                       Ethanol    Drosophila mojavensis         A514   

      dosage age_at_initiation treatment_duration  \
3174   10 µM               NaN                NaN   
0         4%               NaN                NaN   
2613   10 µM              

In [97]:
data['age_at_initiation'].value_counts()

age_at_initiation
12 months              41
after weaning          40
4 months               32
9 months               29
6 months               24
20 months              21
1 month                17
2 months               17
10 months              16
5 months               13
8 months               12
16 months              12
18 months              10
7 months                8
11 months               8
3 months                6
3.5 months              6
1.1 month               4
16-18 months            4
24 months               4
14 months               4
26 months               3
15 months               3
19 months               3
23 months               2
10-12 months            2
5-9 months              2
2.6 months              2
17.2 months             2
13 months               2
8.1 months              2
8.1 or 9.3 months       2
18.5 months             2
9.4 or 10.4 months      2
9.4 months              2
10.4 or 12.1 months     1
19.4 months             1
21 months           

In [101]:
@dataclass
class ParsedTime:
    raw: str
    kept: bool
    keep_reason: str          # simple | milestone | event | reject_*
    value: Optional[float] = None
    unit: Optional[str] = None
    milestone: Optional[str] = None


TIME_UNITS = {
    "day": "Days",
    "days": "Days",
    "week": "Weeks",
    "weeks": "Weeks",
    "month": "Months",
    "months": "Months",
}


def normalize_time_text(x) -> str:
    if x is None:
        return ""
    x = str(x).strip()
    x = x.replace("\u2009", " ").replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    x = x.lower()

    # typo normalization
    x = x.replace("unti death", "until death")
    x = x.replace("26 month", "26 months")

    return x


def parse_age_at_initiation(raw_value) -> ParsedTime:
    raw = normalize_time_text(raw_value)
    if not raw or raw == "nan":
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_empty")

    # explicit milestone whitelist
    if raw == "after weaning":
        return ParsedTime(
            raw=raw,
            kept=True,
            keep_reason="milestone",
            milestone="AfterWeaning",
        )

    # reject non-atomic cases
    if " or " in raw:
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_disjunction")
    if re.search(r"\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?", raw):
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_range")
    if "," in raw:
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_multivalue")

    # exact age in months only
    m = re.fullmatch(r"(\d+(?:\.\d+)?)\s*(month|months)", raw)
    if m:
        return ParsedTime(
            raw=raw,
            kept=True,
            keep_reason="simple",
            value=float(m.group(1)),
            unit="Months",
        )

    return ParsedTime(raw=raw, kept=False, keep_reason="reject_unparsed_age")


def parse_treatment_duration(raw_value) -> ParsedTime:
    raw = normalize_time_text(raw_value)
    if not raw or raw == "nan":
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_empty")

    # explicit event whitelist
    if raw == "until death":
        return ParsedTime(
            raw=raw,
            kept=True,
            keep_reason="event",
            milestone="UntilDeath",
        )

    # reject non-atomic cases
    if " or " in raw:
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_disjunction")
    if re.search(r"\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?", raw):
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_range")
    if "," in raw:
        return ParsedTime(raw=raw, kept=False, keep_reason="reject_multivalue")

    # exact duration
    m = re.fullmatch(r"(\d+(?:\.\d+)?)\s*(day|days|week|weeks|month|months)", raw)
    if m:
        return ParsedTime(
            raw=raw,
            kept=True,
            keep_reason="simple",
            value=float(m.group(1)),
            unit=TIME_UNITS[m.group(2)],
        )

    return ParsedTime(raw=raw, kept=False, keep_reason="reject_unparsed_duration")

def parse_significance(x) -> Optional[str]:
    if x is None:
        return "Unreported"
    t = str(x).strip().upper()
    if not t or t == "NAN":
        return "Unreported"
    if t == "S":
        return "Significant"
    if t == "NS":
        return "NotSignificant"
    return "Unreported"

def metta_time_expr(parsed: ParsedTime) -> Optional[str]:
    if not parsed.kept:
        return None
    if parsed.keep_reason in {"milestone", "event"}:
        return parsed.milestone
    return f"(Measure {parsed.value} {parsed.unit})"

In [102]:
import math
import re
import pandas as pd


def is_missing(x) -> bool:
    return pd.isna(x) or str(x).strip() == ""


def sanitize_symbol(x: str) -> str:
    s = str(x).strip()
    s = s.replace("μ", "u").replace("µ", "u")
    s = s.replace("%", "Percent")
    s = s.replace("+", "Plus")
    s = s.replace("/", "_")
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"[-\s]+", "_", s)
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")
    if not s:
        s = "Unknown"
    if re.match(r"^\d", s):
        s = f"N_{s}"
    return s


def quote_string(x: str) -> str:
    return '"' + str(x).replace("\\", "\\\\").replace('"', '\\"') + '"'


def metta_time_expr(parsed) -> Optional[str]:
    if not parsed.kept:
        return None
    if parsed.milestone:
        return parsed.milestone
    return f"(Measure {parsed.value} {parsed.unit})"


def metta_dose_atoms(row_id: str, parsed) -> list[str]:
    if not parsed.kept:
        return []

    atoms = []
    measure = f"(Measure {parsed.value} {parsed.unit})"

    # medium-bearing dosages are concentrations/exposure in carrier
    if parsed.medium:
        atoms.append(f"(Concentration {row_id} {measure})")
        atoms.append(f"(DeliveryMedium {row_id} {sanitize_symbol(parsed.medium)})")
    else:
        # keep this generic unless your ontology has a better predicate
        atoms.append(f"(Dosage {row_id} {measure})")

    if parsed.frequency:
        atoms.append(f"(AdministrationFrequency {row_id} {sanitize_symbol(parsed.frequency)})")

    return atoms


def build_drugage_metta(df: pd.DataFrame, out_path: str) -> None:
    lines = []
    lines.append("; Auto-generated DrugAge ETL")
    lines.append("")

    for idx, row in df.iterrows():
        row_id = f"DrugAgeRow_{idx}"

        compound = sanitize_symbol(row["compound_name"])
        species = sanitize_symbol(row["species"])
        strain_raw = row.get("strain", None)
        gender_raw = row.get("gender", None)
        pubmed_raw = row.get("pubmed_id", None)
        itp_raw = row.get("ITP", None)

        lines.append(f"; row {idx}")
        lines.append(f"(Inheritance {row_id} Experiment)")
        lines.append(f"(UsesIntervention {row_id} {compound})")
        lines.append(f"(UsesSpecies {row_id} {species})")

        if not is_missing(strain_raw):
            strain = sanitize_symbol(strain_raw)
            lines.append(f"(UsesStrain {row_id} {strain})")

        if not is_missing(gender_raw):
            gender = sanitize_symbol(gender_raw)
            lines.append(f"(HasSex {row_id} {gender})")

        if not is_missing(pubmed_raw):
            try:
                pmid = int(float(pubmed_raw))
                lines.append(f"(ReportedIn {row_id} PMID_{pmid})")
            except Exception:
                lines.append(f"(ReportedIn {row_id} {sanitize_symbol(pubmed_raw)})")

        if not is_missing(itp_raw) and str(itp_raw).strip().lower() == "yes":
            lines.append(f"(IsITPStudy {row_id})")

        # dosage
        dose = parse_dose(row.get("dosage", None))
        lines.extend(metta_dose_atoms(row_id, dose))

        # age at initiation
        age = parse_age_at_initiation(row.get("age_at_initiation", None))
        age_expr = metta_time_expr(age)
        if age_expr is not None:
            lines.append(f"(AgeAtInitiation {row_id} {age_expr})")

        # treatment duration
        dur = parse_treatment_duration(row.get("treatment_duration", None))
        dur_expr = metta_time_expr(dur)
        if dur_expr is not None:
            lines.append(f"(TreatmentDuration {row_id} {dur_expr})")

        # avg lifespan change
        avg_change = row.get("avg_lifespan_change_percent", None)
        if not is_missing(avg_change):
            lines.append(f"(AvgLifespanChangePercent {row_id} {float(avg_change)})")

        avg_sig = parse_significance(row.get("avg_lifespan_significance", None))
        if avg_sig is not None:
            lines.append(f"(AvgLifespanSignificance {row_id} {avg_sig})")

        # max lifespan change
        max_change = row.get("max_lifespan_change_percent", None)
        if not is_missing(max_change):
            lines.append(f"(MaxLifespanChangePercent {row_id} {float(max_change)})")

        max_sig = parse_significance(row.get("max_lifespan_significance", None))
        if max_sig is not None:
            lines.append(f"(MaxLifespanSignificance {row_id} {max_sig})")

        # weight change
        weight_change = row.get("weight_change_percent", None)
        if not is_missing(weight_change):
            lines.append(f"(WeightChangePercent {row_id} {float(weight_change)})")

        weight_sig = parse_significance(row.get("weight_change_significance", None))
        if weight_sig is not None:
            lines.append(f"(WeightChangeSignificance {row_id} {weight_sig})")

        lines.append("")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

In [103]:
build_drugage_metta(data, "drugage_etl.metta")